# AIID PS5 Insight Studio

### One-click, AI-assisted evidence notebook for policy teams, journalists, and researchers

This notebook is designed for **PS5-style decision storytelling** from AI Incident Database snapshots.

What this notebook delivers in one run:
- Loads AIID from `.tar.bz2`, `.tar.gz`, `.tgz`, or `.tz.gz`
- Profiles **all CSV/JSON tables** from the Mongo snapshot
- Uses inferred **primary key / foreign key** links for joins
- Builds full incident analysis across `incidents`, `classifications_*`, `reports`, and related tables
- Generates visualizations with **AI insight for each graph**
- Exports a polished HTML briefing for non-technical audiences


In [ ]:
# Cell 1: Setup, theme, runtime config
import os
import re
import ast
import json
import math
import textwrap
import warnings
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, Any, Optional, List

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown, HTML

warnings.filterwarnings("ignore", category=FutureWarning)


def _load_local_env() -> Optional[Path]:
    """Load .env from cwd or parent so notebook runs from root or notebooks/."""
    candidates = [Path.cwd() / ".env", Path.cwd().parent / ".env"]
    for env_path in candidates:
        if not env_path.exists():
            continue
        try:
            from dotenv import load_dotenv  # type: ignore
            load_dotenv(dotenv_path=env_path, override=False)
            return env_path
        except Exception:
            # Fallback parser when python-dotenv is unavailable in kernel env.
            for raw_line in env_path.read_text(encoding="utf-8").splitlines():
                line = raw_line.strip()
                if (not line) or line.startswith("#") or ("=" not in line):
                    continue
                key, value = line.split("=", 1)
                key = key.strip()
                value = value.strip().strip('"').strip("'")
                if key:
                    os.environ.setdefault(key, value)
            return env_path
    return None


LOADED_ENV_PATH = _load_local_env()

# -------------------- User Config --------------------
SNAPSHOT_URL = "https://pub-72b2b2fc36ec423189843747af98f80e.r2.dev/backup-20260223102103.tar.bz2"
LOCAL_ARCHIVE = None          # Example: "./aiid_cache/backup-20260223102103.tar.bz2"
EXTRACTED_DIR = None          # Example: r"C:\Users\...\mongodump_full_snapshot"

RUN_AI_SUMMARY = True
RUN_AI_INSIGHTS = True
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o")
AI_API_KEY = (os.getenv("OPENAI_API_KEY") or os.getenv("OPENAI_KEY") or "").strip()

CACHE_DIR = Path("./aiid_cache")
REPORTS_DIR = CACHE_DIR / "reports"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Typography + palette
TITLE_FONT = "Merriweather, Georgia, serif"
BODY_FONT = "Source Sans Pro, Segoe UI, Arial, sans-serif"
MONO_FONT = "JetBrains Mono, Consolas, monospace"

PALETTE = {
    "ink": "#14213d",
    "sub": "#4a5d70",
    "bg": "#f4f8fc",
    "accent": "#0f4c81",
    "accent2": "#2c7da0",
    "good": "#1b9e77",
    "warn": "#e76f51",
    "bar": ["#0f4c81", "#2c7da0", "#468faf", "#61a5c2", "#89c2d9", "#a9d6e5"]
}

px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = PALETTE["bar"]


def style_fig(fig: go.Figure, title: str, subtitle: str = "") -> go.Figure:
    fig.update_layout(
        title=dict(
            text=f"<b>{title}</b><br><span style='font-size:13px;color:{PALETTE['sub']}'>{subtitle}</span>",
            x=0.01,
            xanchor="left",
            font=dict(family=TITLE_FONT, size=24, color=PALETTE["ink"]),
        ),
        font=dict(family=BODY_FONT, size=13, color=PALETTE["ink"]),
        paper_bgcolor=PALETTE["bg"],
        plot_bgcolor="white",
        margin=dict(l=50, r=30, t=95, b=50),
        legend=dict(title=None),
    )
    fig.update_xaxes(gridcolor="rgba(20,33,61,0.10)", zeroline=False)
    fig.update_yaxes(gridcolor="rgba(20,33,61,0.10)", zeroline=False)
    return fig


def card(label: str, value: str, note: str = ""):
    return f"""
    <div style='
        background:white;
        border:1px solid #d9e3ef;
        border-radius:14px;
        padding:14px 16px;
        box-shadow:0 6px 14px rgba(20,33,61,.06);
        min-width:220px;'>
      <div style='font-family:{BODY_FONT};color:{PALETTE['sub']};font-size:12px;text-transform:uppercase;letter-spacing:.05em'>{label}</div>
      <div style='font-family:{TITLE_FONT};font-size:30px;color:{PALETTE['ink']};line-height:1.2'>{value}</div>
      <div style='font-family:{BODY_FONT};font-size:12px;color:{PALETTE['sub']}'>{note}</div>
    </div>
    """


CHART_INSIGHTS: Dict[str, Dict[str, Any]] = {}
CHART_FILES: Dict[str, str] = {}


def register_chart(chart_key: str, file_path: Path):
    CHART_FILES[chart_key] = str(file_path)


def ai_insight(title: str, facts: Dict[str, Any], fallback: str) -> str:
    if not (RUN_AI_INSIGHTS and AI_API_KEY):
        return fallback
    try:
        from openai import OpenAI
        client = OpenAI(api_key=AI_API_KEY)
        system = "You are an AI policy analyst. Return 4 concise markdown bullets using only the provided facts. Include concrete numbers."
        user = f"Chart: {title}\nFacts JSON:\n{json.dumps(facts, ensure_ascii=False)}"
        resp = client.chat.completions.create(
            model=OPENAI_MODEL,
            temperature=0.2,
            max_tokens=320,
            messages=[{"role":"system","content":system},{"role":"user","content":user}],
        )
        text = (resp.choices[0].message.content or "").strip()
        return text if text else fallback
    except Exception:
        return fallback


print("✅ Environment configured")
print(f"🧾 Env file: {LOADED_ENV_PATH.resolve() if LOADED_ENV_PATH else 'not found'}")
print(f"📁 Cache dir: {CACHE_DIR.resolve()}")
print(f"🤖 OpenAI model: {OPENAI_MODEL}")
print(f"🔑 OpenAI key detected: {'yes' if bool(AI_API_KEY) else 'no'}")


## Data Loading

The loader accepts snapshot archives directly and performs:
1. Secure extraction
2. Smart file discovery (`incidents`, `classifications`, `entities`)
3. Normalization for analysis-ready columns

Supported archives: `.tar.bz2`, `.tbz2`, `.tar.gz`, `.tgz`, `.tz.gz`, `.tar`



In [ ]:
# Cell 2: AIID ingestion + multi-table relationship helpers
import sys

backend_path = str(Path("backend").resolve())
if backend_path not in sys.path:
    sys.path.append(backend_path)

from app.notebooks.aiid_snapshot_utils import (
    resolve_snapshot as resolve_snapshot_source,
    load_tabular_tables,
    infer_relationships,
    build_canonical_incident_df,
)


def split_multi(value: Any) -> List[str]:
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except Exception:
        pass
    text = str(value).strip()
    if not text:
        return []

    vals = []
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, (list, tuple, set)):
                vals = [str(v) for v in parsed]
            else:
                vals = [text]
        except Exception:
            vals = [text]
    else:
        vals = [text]

    out = []
    for v in vals:
        for part in re.split(r"\s*[,;|/]\s*", str(v).strip()):
            if part:
                out.append(part)
    return out


def top_counts_multi(frame: pd.DataFrame, col: str, n: int = 12) -> pd.DataFrame:
    if col not in frame.columns:
        return pd.DataFrame(columns=["label", "count", "pct"])
    items = []
    for value in frame[col].dropna().tolist():
        items.extend(split_multi(value))
    if not items:
        return pd.DataFrame(columns=["label", "count", "pct"])
    vc = pd.Series(items, dtype="string").value_counts().head(n)
    out = vc.rename_axis("label").reset_index(name="count")
    out["pct"] = (out["count"] / max(int(vc.sum()), 1) * 100).round(1)
    return out


In [ ]:
# Cell 3: Load snapshot (URL/local archive/extracted), profile all tables, infer PK/FK, build canonical frame

snapshot_root, file_inventory = resolve_snapshot_source(
    snapshot_url=SNAPSHOT_URL,
    local_archive=LOCAL_ARCHIVE,
    extracted_dir=EXTRACTED_DIR,
    cache_dir=CACHE_DIR,
)

all_tables, table_catalog = load_tabular_tables(snapshot_root)
relationship_df = infer_relationships(all_tables)

df = build_canonical_incident_df(file_inventory)

print(f"✅ Snapshot root: {snapshot_root}")
print(f"✅ Tabular tables loaded: {len(all_tables):,}")
print(f"✅ PK/FK edges inferred: {len(relationship_df):,}")
print(f"✅ Canonical incident frame: {len(df):,} rows × {df.shape[1]:,} columns")

kpi_html = "".join([
    card("Tabular Tables", f"{len(all_tables):,}", "CSV/JSON files discovered in snapshot"),
    card("PK/FK Edges", f"{len(relationship_df):,}", "Inferred relationship links"),
    card("Incident Rows", f"{len(df):,}", "Canonical AIID incident records"),
    card("Incident Columns", f"{df.shape[1]:,}", "Merged multi-table schema"),
])
display(HTML(f"<div style='display:flex;flex-wrap:wrap;gap:12px'>{kpi_html}</div>"))

if not table_catalog.empty:
    display(Markdown("### Snapshot Table Catalog"))
    display(table_catalog.head(30))

if not relationship_df.empty:
    display(Markdown("### Inferred Primary Key / Foreign Key Links"))
    display(relationship_df.head(30))

display(df.head(3))


if not table_catalog.empty:
    _tc = table_catalog[table_catalog["status"] == "loaded"].copy()
    _tc = _tc.dropna(subset=["rows"]).sort_values("rows", ascending=False).head(20)
    if not _tc.empty:
        fig_tables = px.bar(
            _tc.sort_values("rows"),
            x="rows", y="table", orientation="h", text="columns",
            labels={"rows": "Rows", "table": "Table", "columns": "Columns"},
            color="rows", color_continuous_scale="Blues",
        )
        fig_tables.update_traces(texttemplate="cols=%{text}", textposition="outside")
        style_fig(fig_tables, "Snapshot Table Landscape", "Top loaded tables by row count")
        fig_tables.show()
        fig_tables.write_html(REPORTS_DIR / "fig_table_landscape.html", include_plotlyjs="cdn")
        register_chart("table_landscape", REPORTS_DIR / "fig_table_landscape.html")

        tbl_facts = {
            "tables": int(len(_tc)),
            "largest_table": str(_tc.iloc[0]["table"]),
            "largest_rows": int(_tc.iloc[0]["rows"]),
        }
        tbl_fb = (
            "### AI Insight - Snapshot Table Landscape\n"
            f"- Top loaded view contains {tbl_facts['tables']} tables.\n"
            f"- Largest table is {tbl_facts['largest_table']} with {tbl_facts['largest_rows']:,} rows.\n"
            "- Row concentration indicates where most analytical signal resides.\n"
            "- Smaller tables still matter for enrichment and relationship joins."
        )
        tbl_md = ai_insight("Snapshot Table Landscape", tbl_facts, tbl_fb)
        CHART_INSIGHTS["table_landscape"] = {"title": "Snapshot Table Landscape", "facts": tbl_facts, "insight": tbl_md}
        display(Markdown(tbl_md))


if not relationship_df.empty:
    rel = relationship_df.head(40).copy()
    left_nodes = rel["source_table"].astype(str).unique().tolist()
    right_nodes = rel["target_table"].astype(str).unique().tolist()
    nodes = left_nodes + [n for n in right_nodes if n not in left_nodes]
    node_ix = {n: i for i, n in enumerate(nodes)}

    fig_rel = go.Figure(data=[go.Sankey(
        node=dict(
            label=nodes,
            pad=14,
            thickness=14,
            line=dict(color="rgba(20,33,61,.25)", width=0.5),
            color=["#2c7da0" if n in left_nodes else "#e76f51" for n in nodes],
        ),
        link=dict(
            source=rel["source_table"].map(node_ix),
            target=rel["target_table"].map(node_ix),
            value=rel["matched_values"].clip(lower=1),
            color="rgba(70,143,175,0.28)",
        ),
    )])
    style_fig(fig_rel, "PK/FK Relationship Flow", "Inferred join structure across snapshot tables")
    fig_rel.show()
    fig_rel.write_html(REPORTS_DIR / "fig_pk_fk_flow.html", include_plotlyjs="cdn")
    register_chart("pk_fk_flow", REPORTS_DIR / "fig_pk_fk_flow.html")

    rel_facts = {
        "edges": int(len(relationship_df)),
        "best_overlap_pct": float(relationship_df["overlap_pct"].max()),
        "top_edge": relationship_df.head(1).to_dict(orient="records")[0],
    }
    rel_fb = (
        "### AI Insight - PK/FK Relationship Flow\n"
        f"- Inferred {rel_facts['edges']} relationship edges across loaded tables.\n"
        f"- Highest observed overlap is {rel_facts['best_overlap_pct']:.2f}%.\n"
        f"- Strongest edge: {rel_facts['top_edge']}.\n"
        "- These links define the trust boundary for multi-table joins in analysis."
    )
    rel_md = ai_insight("PK/FK Relationship Flow", rel_facts, rel_fb)
    CHART_INSIGHTS["pk_fk_flow"] = {"title": "PK/FK Relationship Flow", "facts": rel_facts, "insight": rel_md}
    display(Markdown(rel_md))


In [ ]:
# Cell 4: Canonical fields and KPI cards for non-technical audiences

def pick_col(frame: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    cols = set(frame.columns)
    for c in candidates:
        if c in cols:
            return c
    return None

col_date = pick_col(df, ["date", "incident_date"])
col_year = pick_col(df, ["year"])
col_month = pick_col(df, ["month", "month_name"])
col_harm = pick_col(df, ["harm_type", "risk_domain", "harm_domain"])
col_sector = pick_col(df, ["sector_of_deployment", "infrastructure_sectors", "sector"])
col_deployer = pick_col(df, ["alleged_deployer_primary", "allegeddeployerofaisystem_primary", "alleged_deployer", "allegeddeployerofaisystem"])
col_developer = pick_col(df, ["alleged_developer_primary", "allegeddeveloperofaisystem_primary", "alleged_developer", "allegeddeveloperofaisystem"])
col_title = pick_col(df, ["title", "incident_title"])

if col_date and not np.issubdtype(df[col_date].dtype, np.datetime64):
    df[col_date] = pd.to_datetime(df[col_date], errors="coerce")
if col_year is None and col_date:
    df["year"] = df[col_date].dt.year
    col_year = "year"
if col_month is None and col_date:
    df["month"] = df[col_date].dt.month
    col_month = "month"

coverage = {
    "harm": float(df[col_harm].notna().mean() * 100) if col_harm else 0,
    "sector": float(df[col_sector].notna().mean() * 100) if col_sector else 0,
    "deployer": float(df[col_deployer].notna().mean() * 100) if col_deployer else 0,
    "developer": float(df[col_developer].notna().mean() * 100) if col_developer else 0,
}

date_min = df[col_date].min() if col_date else None
date_max = df[col_date].max() if col_date else None

kpi_html = "".join([
    card("Total Incidents", f"{len(df):,}", "Records in current snapshot"),
    card("Date Range", f"{date_min.date() if pd.notna(date_min) else 'N/A'} → {date_max.date() if pd.notna(date_max) else 'N/A'}", "Earliest to latest incident date"),
    card("Harm Coverage", f"{coverage['harm']:.1f}%", "Rows with harm classification"),
    card("Sector Coverage", f"{coverage['sector']:.1f}%", "Rows with sector classification"),
])

display(HTML(f"<div style='display:flex;flex-wrap:wrap;gap:12px'>{kpi_html}</div>"))


In [ ]:
# Cell 5: Incident trend over time (policy-ready growth and volatility view)
if not col_year:
    raise ValueError("No year column available for trend analysis")

yearly = (
    df.dropna(subset=[col_year])
      .assign(year_num=lambda x: pd.to_numeric(x[col_year], errors="coerce"))
      .dropna(subset=["year_num"])
)
yearly["year_num"] = yearly["year_num"].astype(int)
yr = yearly.groupby("year_num").size().reset_index(name="incidents").sort_values("year_num")
yr["rolling_3y"] = yr["incidents"].rolling(3, min_periods=1).mean()
yr["cumulative"] = yr["incidents"].cumsum()

fig_trend = make_subplots(specs=[[{"secondary_y": True}]])
fig_trend.add_trace(
    go.Scatter(
        x=yr["year_num"], y=yr["incidents"], mode="lines+markers",
        name="Incidents per year", line=dict(color=PALETTE["accent"], width=3),
        marker=dict(size=7)
    ),
    secondary_y=False,
)
fig_trend.add_trace(
    go.Scatter(
        x=yr["year_num"], y=yr["rolling_3y"], mode="lines",
        name="3-year moving average", line=dict(color=PALETTE["warn"], width=2, dash="dash")
    ),
    secondary_y=False,
)
fig_trend.add_trace(
    go.Bar(
        x=yr["year_num"], y=yr["cumulative"], name="Cumulative incidents",
        marker_color="rgba(44,125,160,0.22)", opacity=0.55
    ),
    secondary_y=True,
)

fig_trend.update_yaxes(title_text="Yearly incidents", secondary_y=False)
fig_trend.update_yaxes(title_text="Cumulative incidents", secondary_y=True)
style_fig(fig_trend, "AI Incident Momentum", "Growth, smoothing, and cumulative trajectory")
fig_trend.show()

fig_trend.write_html(REPORTS_DIR / "fig_trend_momentum.html", include_plotlyjs="cdn")



register_chart("trend_momentum", REPORTS_DIR / "fig_trend_momentum.html")

trend_facts = {
    "years": int(len(yr)),
    "first_year": int(yr.iloc[0]["year_num"]) if len(yr) else None,
    "last_year": int(yr.iloc[-1]["year_num"]) if len(yr) else None,
    "first_count": int(yr.iloc[0]["incidents"]) if len(yr) else 0,
    "last_count": int(yr.iloc[-1]["incidents"]) if len(yr) else 0,
    "peak_year": int(yr.loc[yr["incidents"].idxmax()]["year_num"]) if len(yr) else None,
    "peak_count": int(yr["incidents"].max()) if len(yr) else 0,
}
trend_fb = (
    "### AI Insight - AI Incident Momentum\n"
    f"- Coverage spans {trend_facts['years']} years.\n"
    f"- Volume moved from {trend_facts['first_count']:,} to {trend_facts['last_count']:,} incidents.\n"
    f"- Peak year is {trend_facts['peak_year']} with {trend_facts['peak_count']:,} incidents.\n"
    "- Treat recent-year drops with reporting-lag caution."
)
trend_md = ai_insight("AI Incident Momentum", trend_facts, trend_fb)
CHART_INSIGHTS["trend_momentum"] = {"title": "AI Incident Momentum", "facts": trend_facts, "insight": trend_md}
display(Markdown(trend_md))


In [ ]:
# Cell 6: Harm profile and sector exposure (dual audience charting)

harm_top = top_counts_multi(df, col_harm, 12) if col_harm else pd.DataFrame(columns=["label","count","pct"])
sector_top = top_counts_multi(df, col_sector, 12) if col_sector else pd.DataFrame(columns=["label","count","pct"])

if not harm_top.empty:
    fig_harm = px.bar(
        harm_top.sort_values("count"),
        x="count", y="label", orientation="h", text="pct",
        labels={"count": "Incidents", "label": "Harm Type"},
        color="count", color_continuous_scale="Blues"
    )
    fig_harm.update_traces(texttemplate="%{text}%", textposition="outside")
    style_fig(fig_harm, "Harm Type Distribution", "Top classified harms in this snapshot")
    fig_harm.show()
    fig_harm.write_html(REPORTS_DIR / "fig_harm_distribution.html", include_plotlyjs="cdn")
    register_chart("harm_distribution", REPORTS_DIR / "fig_harm_distribution.html")

    harm_facts = {
        "categories": int(len(harm_top)),
        "top_harm": str(harm_top.iloc[0]["label"]),
        "top_count": int(harm_top.iloc[0]["count"]),
        "coverage_pct": float(coverage.get("harm", 0.0)),
    }
    harm_fb = (
        "### AI Insight - Harm Type Distribution\n"
        f"- Top harm is {harm_facts['top_harm']} ({harm_facts['top_count']:,} incidents).\n"
        f"- Harm coverage is {harm_facts['coverage_pct']:.1f}% of incident rows.\n"
        f"- Top view includes {harm_facts['categories']} harm categories.\n"
        "- Prioritize safeguards for highest-frequency harm classes while improving missing-label coverage."
    )
    harm_md = ai_insight("Harm Type Distribution", harm_facts, harm_fb)
    CHART_INSIGHTS["harm_distribution"] = {"title": "Harm Type Distribution", "facts": harm_facts, "insight": harm_md}
    display(Markdown(harm_md))
else:
    print("⚠️ Skipping harm chart: no data")

if not sector_top.empty:
    fig_sector = px.treemap(
        sector_top,
        path=[px.Constant("All sectors"), "label"],
        values="count",
        color="count",
        color_continuous_scale="Tealgrn",
    )
    style_fig(fig_sector, "Sector Exposure Map", "Treemap of sectors where incidents are most concentrated")
    fig_sector.show()
    fig_sector.write_html(REPORTS_DIR / "fig_sector_treemap.html", include_plotlyjs="cdn")
    register_chart("sector_treemap", REPORTS_DIR / "fig_sector_treemap.html")

    sec_facts = {
        "categories": int(len(sector_top)),
        "top_sector": str(sector_top.iloc[0]["label"]),
        "top_count": int(sector_top.iloc[0]["count"]),
        "coverage_pct": float(coverage.get("sector", 0.0)),
    }
    sec_fb = (
        "### AI Insight - Sector Exposure Map\n"
        f"- Top sector is {sec_facts['top_sector']} ({sec_facts['top_count']:,} incidents).\n"
        f"- Sector coverage is {sec_facts['coverage_pct']:.1f}% of incident rows.\n"
        f"- Top view includes {sec_facts['categories']} sectors.\n"
        "- Sector concentration helps prioritize domain-specific governance and auditing."
    )
    sec_md = ai_insight("Sector Exposure Map", sec_facts, sec_fb)
    CHART_INSIGHTS["sector_treemap"] = {"title": "Sector Exposure Map", "facts": sec_facts, "insight": sec_md}
    display(Markdown(sec_md))
else:
    print("⚠️ Skipping sector chart: no data")


In [ ]:
# Cell 7: Harm × Sector risk matrix heatmap
if col_harm and col_sector:
    rows = []
    for _, r in df[[col_harm, col_sector]].dropna().iterrows():
        harms = split_multi(r[col_harm])
        sectors = split_multi(r[col_sector])
        for h in harms[:4]:
            for s in sectors[:4]:
                rows.append((h, s))

    hs = pd.DataFrame(rows, columns=["harm", "sector"])
    if hs.empty:
        print("⚠️ Skipping heatmap: no valid harm/sector intersections")
    else:
        top_h = hs["harm"].value_counts().head(10).index
        top_s = hs["sector"].value_counts().head(10).index
        hs = hs[hs["harm"].isin(top_h) & hs["sector"].isin(top_s)]

        pivot = pd.crosstab(hs["harm"], hs["sector"])
        fig_heat = px.imshow(
            pivot,
            text_auto=True,
            color_continuous_scale="YlOrRd",
            aspect="auto",
            labels=dict(x="Sector", y="Harm Type", color="Incidents")
        )
        style_fig(fig_heat, "Risk Interaction Matrix", "Where specific harms and sectors intersect most")
        fig_heat.show()
        fig_heat.write_html(REPORTS_DIR / "fig_risk_heatmap.html", include_plotlyjs="cdn")
        register_chart("risk_heatmap", REPORTS_DIR / "fig_risk_heatmap.html")

        mx = pivot.stack().sort_values(ascending=False).head(1)
        if len(mx):
            (mx_h, mx_s), mx_v = mx.index[0], int(mx.iloc[0])
        else:
            mx_h, mx_s, mx_v = None, None, 0

        heat_facts = {
            "harm_nodes": int(pivot.shape[0]),
            "sector_nodes": int(pivot.shape[1]),
            "max_harm": mx_h,
            "max_sector": mx_s,
            "max_count": mx_v,
            "intersection_total": int(pivot.values.sum()),
        }
        heat_fb = (
            "### AI Insight - Risk Interaction Matrix\n"
            f"- Matrix maps {heat_facts['harm_nodes']} harms across {heat_facts['sector_nodes']} sectors.\n"
            f"- Strongest intersection is {heat_facts['max_harm']} × {heat_facts['max_sector']} ({heat_facts['max_count']:,}).\n"
            f"- Total counted intersections in top matrix: {heat_facts['intersection_total']:,}.\n"
            "- Cross-domain intersections indicate where targeted controls can reduce multiple harms." 
        )
        heat_md = ai_insight("Risk Interaction Matrix", heat_facts, heat_fb)
        CHART_INSIGHTS["risk_heatmap"] = {"title": "Risk Interaction Matrix", "facts": heat_facts, "insight": heat_md}
        display(Markdown(heat_md))
else:
    print("⚠️ Skipping heatmap: required columns unavailable")


In [ ]:
# Cell 8: Diagram view — Deployer → Harm Sankey
if col_deployer and col_harm:
    sankey_df = (
        df[[col_deployer, col_harm]]
        .dropna()
        .astype(str)
    )

    top_dep = sankey_df[col_deployer].value_counts().head(12).index
    top_hm = sankey_df[col_harm].value_counts().head(10).index
    sankey_df = sankey_df[sankey_df[col_deployer].isin(top_dep) & sankey_df[col_harm].isin(top_hm)]

    link_df = sankey_df.groupby([col_deployer, col_harm]).size().reset_index(name="value")

    left_nodes = list(link_df[col_deployer].unique())
    right_nodes = list(link_df[col_harm].unique())
    nodes = left_nodes + right_nodes
    node_index = {n: i for i, n in enumerate(nodes)}

    source = link_df[col_deployer].map(node_index)
    target = link_df[col_harm].map(node_index)

    fig_sankey = go.Figure(
        data=[
            go.Sankey(
                node=dict(
                    label=nodes,
                    pad=14,
                    thickness=14,
                    line=dict(color="rgba(20,33,61,.25)", width=0.5),
                    color=["#2c7da0"] * len(left_nodes) + ["#e76f51"] * len(right_nodes),
                ),
                link=dict(
                    source=source,
                    target=target,
                    value=link_df["value"],
                    color="rgba(70,143,175,0.28)",
                ),
            )
        ]
    )
    style_fig(fig_sankey, "Responsibility Flow Diagram", "How deployers connect to the most reported harm types")
    fig_sankey.show()
    fig_sankey.write_html(REPORTS_DIR / "fig_deployer_harm_sankey.html", include_plotlyjs="cdn")
    register_chart("deployer_harm_sankey", REPORTS_DIR / "fig_deployer_harm_sankey.html")

    top_link = link_df.sort_values("value", ascending=False).head(1).to_dict(orient="records")
    sankey_facts = {
        "nodes": int(len(nodes)),
        "links": int(len(link_df)),
        "total_flow": int(link_df["value"].sum()),
        "top_link": top_link[0] if top_link else None,
    }
    sankey_fb = (
        "### AI Insight - Responsibility Flow Diagram\n"
        f"- Diagram contains {sankey_facts['nodes']} nodes and {sankey_facts['links']} deployer-harm links.\n"
        f"- Total mapped flow count is {sankey_facts['total_flow']:,}.\n"
        f"- Strongest flow is {sankey_facts['top_link']}.\n"
        "- Concentrated flow pathways are priority zones for governance and accountability."
    )
    sankey_md = ai_insight("Responsibility Flow Diagram", sankey_facts, sankey_fb)
    CHART_INSIGHTS["deployer_harm_sankey"] = {"title": "Responsibility Flow Diagram", "facts": sankey_facts, "insight": sankey_md}
    display(Markdown(sankey_md))
else:
    print("⚠️ Skipping Sankey: required columns unavailable")


In [ ]:
# Cell 9: Seasonality and reporting rhythm
if col_date is not None:
    ts = df[[col_date]].dropna().copy()
    ts["month_start"] = pd.to_datetime(ts[col_date], errors="coerce").dt.to_period("M").dt.to_timestamp()
    monthly = ts.groupby("month_start").size().reset_index(name="incidents")
    monthly["rolling_6m"] = monthly["incidents"].rolling(6, min_periods=1).mean()

    fig_monthly = go.Figure()
    fig_monthly.add_trace(go.Bar(x=monthly["month_start"], y=monthly["incidents"], name="Monthly incidents", marker_color="#89c2d9"))
    fig_monthly.add_trace(go.Scatter(x=monthly["month_start"], y=monthly["rolling_6m"], name="6-month rolling average", line=dict(color="#0f4c81", width=3)))

    style_fig(fig_monthly, "Monthly Incident Rhythm", "Signal smoothing for policy timing and media monitoring")
    fig_monthly.show()
    fig_monthly.write_html(REPORTS_DIR / "fig_monthly_rhythm.html", include_plotlyjs="cdn")
    register_chart("monthly_rhythm", REPORTS_DIR / "fig_monthly_rhythm.html")

    peak = monthly.loc[monthly["incidents"].idxmax()] if len(monthly) else None
    month_facts = {
        "months": int(len(monthly)),
        "peak_month": str(peak["month_start"].date()) if peak is not None else None,
        "peak_count": int(peak["incidents"]) if peak is not None else 0,
        "latest_count": int(monthly.iloc[-1]["incidents"]) if len(monthly) else 0,
    }
    month_fb = (
        "### AI Insight - Monthly Incident Rhythm\n"
        f"- Timeline spans {month_facts['months']} months.\n"
        f"- Peak month is {month_facts['peak_month']} ({month_facts['peak_count']:,} incidents).\n"
        f"- Latest month in view records {month_facts['latest_count']:,} incidents.\n"
        "- Month-level rhythm is useful for watchdog planning and communications timing."
    )
    month_md = ai_insight("Monthly Incident Rhythm", month_facts, month_fb)
    CHART_INSIGHTS["monthly_rhythm"] = {"title": "Monthly Incident Rhythm", "facts": month_facts, "insight": month_md}
    display(Markdown(month_md))
else:
    print("⚠️ Skipping monthly chart: no date field found")

if "report_sources" in df.columns and df["report_sources"].notna().any():
    src_vals = []
    for val in df["report_sources"].dropna().tolist():
        src_vals.extend(split_multi(val))
    src = pd.Series(src_vals, dtype="string")
    src_top = src.value_counts().head(15).rename_axis("source").reset_index(name="mentions")
    if not src_top.empty:
        fig_src = px.bar(
            src_top.sort_values("mentions"),
            x="mentions", y="source", orientation="h",
            labels={"mentions": "Linked reports", "source": "Source domain"},
            color="mentions", color_continuous_scale="Blues",
        )
        style_fig(fig_src, "Reporting Source Footprint", "Domains most frequently linked to incidents")
        fig_src.show()
        fig_src.write_html(REPORTS_DIR / "fig_source_footprint.html", include_plotlyjs="cdn")
        register_chart("source_footprint", REPORTS_DIR / "fig_source_footprint.html")

        src_facts = {
            "sources": int(len(src_top)),
            "top_source": str(src_top.iloc[0]["source"]),
            "top_mentions": int(src_top.iloc[0]["mentions"]),
        }
        src_fb = (
            "### AI Insight - Reporting Source Footprint\n"
            f"- Top source is {src_facts['top_source']} with {src_facts['top_mentions']:,} linked reports.\n"
            f"- Top view includes {src_facts['sources']} source domains.\n"
            "- Source concentration can shape which harms are visible in public evidence.\n"
            "- Use this to contextualize media/reporting bias in trend interpretation."
        )
        src_md = ai_insight("Reporting Source Footprint", src_facts, src_fb)
        CHART_INSIGHTS["source_footprint"] = {"title": "Reporting Source Footprint", "facts": src_facts, "insight": src_md}
        display(Markdown(src_md))


In [ ]:
# Cell 10: Fact pack for AI + reproducible numeric context

def top_list(frame: pd.DataFrame, col: Optional[str], n: int = 8):
    if not col or col not in frame.columns:
        return []
    vals = []
    for v in frame[col].dropna().tolist():
        vals.extend(split_multi(v))
    if not vals:
        return []
    vc = pd.Series(vals, dtype="string").value_counts().head(n)
    total = int(vc.sum())
    return [
        {
            "label": k,
            "count": int(v),
            "pct_within_top": round((int(v) / total) * 100, 2) if total else 0,
        }
        for k, v in vc.items()
    ]

facts = {
    "snapshot_generated_at": datetime.utcnow().isoformat() + "Z",
    "snapshot_source": {
        "snapshot_url": SNAPSHOT_URL,
        "local_archive": LOCAL_ARCHIVE,
        "extracted_dir": EXTRACTED_DIR or str(snapshot_root),
    },
    "rows": int(len(df)),
    "columns": int(df.shape[1]),
    "date_range": {
        "start": str(df[col_date].min().date()) if col_date and pd.notna(df[col_date].min()) else None,
        "end": str(df[col_date].max().date()) if col_date and pd.notna(df[col_date].max()) else None,
    },
    "coverage_pct": {k: round(v, 2) for k, v in coverage.items()},
    "top_harm_types": top_list(df, col_harm, 10),
    "top_sectors": top_list(df, col_sector, 10),
    "top_deployers": top_list(df, col_deployer, 10),
    "top_developers": top_list(df, col_developer, 10),
    "table_catalog": table_catalog.to_dict(orient="records") if 'table_catalog' in globals() else [],
    "relationships": relationship_df.to_dict(orient="records") if 'relationship_df' in globals() else [],
    "chart_insights": CHART_INSIGHTS,
    "chart_files": CHART_FILES,
}

facts_path = (CACHE_DIR / "AIID_Fact_Pack.json").resolve()
print(f"✅ Fact pack saved: {facts_path}")

if 'relationship_df' in globals() and not relationship_df.empty:
    rel_path = (CACHE_DIR / "AIID_Relationships.csv").resolve()
    relationship_df.to_csv(rel_path, index=False)
    print(f"✅ Relationship map saved: {rel_path}")

display(Markdown("```json\n" + json.dumps(facts, indent=2)[:3600] + "\n```"))


In [ ]:
# Cell 11: OpenAI-generated executive narrative (grounded in fact pack)
ai_summary_md = ""


def local_fallback_summary(f: Dict[str, Any]) -> str:
    harms = f.get("top_harm_types", [])[:3]
    sectors = f.get("top_sectors", [])[:3]
    deps = f.get("top_deployers", [])[:3]
    devs = f.get("top_developers", [])[:3]

    def line(items):
        if not items:
            return "No reliable signal in this field."
        return "; ".join([f"{x['label']} ({x['count']})" for x in items])

    return f"""
## Executive Overview
- The snapshot contains **{f.get('rows',0):,} incidents** across **{f.get('columns',0)} fields**.
- Date window: **{f.get('date_range',{}).get('start')} → {f.get('date_range',{}).get('end')}**.
- PK/FK edges inferred: **{len(f.get('relationships', [])):,}**.

## What Stands Out
- Top harms: {line(harms)}
- Top sectors: {line(sectors)}
- Top deployers: {line(deps)}
- Top developers: {line(devs)}

## Policy and Public-interest Lens
- Focus interventions where **harm concentration** and **sector concentration** overlap.
- Treat low-coverage fields as underreported risk, not absence of risk.
- Validate high-stakes findings against relationship lineage and source links.
""".strip()


if RUN_AI_SUMMARY and AI_API_KEY:
    try:
        from openai import OpenAI

        client = OpenAI(api_key=AI_API_KEY)
        system_prompt = """
You are an expert AI policy analyst writing for non-technical decision makers.
Use only the provided facts and chart insights. Do not invent numbers.
If a number is missing, explicitly say data is unavailable.
Return Markdown with sections:
1) Executive Summary
2) Key Quantitative Findings
3) Policy Implications (5 bullets)
4) Journalist Angles (5 bullets)
5) Responsible Interpretation / Limits
Tone: clear, evidence-based, concise but insightful.
""".strip()

        user_prompt = (
            "FACT PACK (JSON):\n" + json.dumps(facts, ensure_ascii=False)
            + "\n\nCHART INSIGHTS (JSON):\n" + json.dumps(CHART_INSIGHTS, ensure_ascii=False)
        )

        resp = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.25,
            max_tokens=1500,
        )
        ai_summary_md = (resp.choices[0].message.content or "").strip()
    except Exception as ex:
        print(f"⚠️ OpenAI summary failed, using local fallback: {ex}")
        ai_summary_md = local_fallback_summary(facts)
else:
    ai_summary_md = local_fallback_summary(facts)

summary_txt = CACHE_DIR / "AIID_Summary_Report.txt"
summary_md = CACHE_DIR / "AIID_Summary_Report.md"
summary_txt.write_text(ai_summary_md, encoding="utf-8")
summary_md.write_text(ai_summary_md, encoding="utf-8")

print(f"✅ Summary saved: {summary_txt}")
print(f"✅ Markdown summary saved: {summary_md}")

display(Markdown(ai_summary_md))


In [ ]:
# Cell 12: Build a polished one-file HTML briefing (charts + AI narrative + chart insights)

def fig_to_html(path: Path) -> str:
    if not path.exists():
        return ""
    return path.read_text(encoding="utf-8")


def md_to_html(md_text: str) -> str:
    try:
        import markdown as md_pkg
        return md_pkg.markdown(md_text, extensions=["tables", "fenced_code"])
    except Exception:
        return "<pre style='white-space:pre-wrap'>" + md_text.replace("<", "&lt;").replace(">", "&gt;") + "</pre>"

summary_html = md_to_html(ai_summary_md)

chart_order = [
    ("table_landscape", "Snapshot Table Landscape"),
    ("pk_fk_flow", "PK/FK Relationship Flow"),
    ("trend_momentum", "AI Incident Momentum"),
    ("harm_distribution", "Harm Type Distribution"),
    ("sector_treemap", "Sector Exposure Map"),
    ("risk_heatmap", "Risk Interaction Matrix"),
    ("deployer_harm_sankey", "Responsibility Flow Diagram"),
    ("monthly_rhythm", "Monthly Incident Rhythm"),
    ("source_footprint", "Reporting Source Footprint"),
]

chart_sections = []
for key, title in chart_order:
    pth = CHART_FILES.get(key)
    if not pth:
        continue
    chart_html = fig_to_html(Path(pth))
    insight_md = CHART_INSIGHTS.get(key, {}).get("insight", "")
    insight_html = md_to_html(insight_md) if insight_md else ""
    chart_sections.append(f"""
    <section class='section'>
      <h2>{title}</h2>
      <div class='insight'>{insight_html}</div>
      {chart_html}
    </section>
    """)

report_html = f"""
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <title>AIID Executive Visual Brief</title>
  <style>
    @import url('https://fonts.googleapis.com/css2?family=Merriweather:wght@700;900&family=Source+Sans+3:wght@400;600;700&family=JetBrains+Mono:wght@400;600&display=swap');
    :root {{ --bg:#f4f8fc; --card:#fff; --ink:#13293d; --sub:#50667a; --line:#d9e3ef; --accent:#0f4c81; }}
    body {{ margin:0; background: radial-gradient(circle at 12% 0%, #eaf4ff 0%, var(--bg) 43%); color:var(--ink); font-family:'Source Sans 3','Segoe UI',Arial,sans-serif; }}
    .wrap {{ max-width:1180px; margin: 0 auto; padding: 20px 14px 40px; }}
    .hero {{ background:var(--card); border:1px solid var(--line); border-radius:18px; padding:22px 24px; box-shadow:0 10px 22px rgba(19,41,61,.08); }}
    h1,h2,h3 {{ font-family:'Merriweather',Georgia,serif; color:#0e2f4d; }}
    h1 {{ margin:.1em 0 .35em; font-size:2rem; }}
    h2 {{ margin-top:1.2em; border-bottom:1px solid #e6eef7; padding-bottom:.35em; }}
    .sub {{ color:var(--sub); font-size:1.05rem; }}
    .section {{ margin-top:16px; background:var(--card); border:1px solid var(--line); border-radius:16px; padding:18px; }}
    .insight {{ background:#f8fbff; border:1px solid #dce8f5; border-radius:10px; padding:10px 12px; margin-bottom:10px; }}
    .foot {{ margin-top:14px; color:var(--sub); font-size:.92rem; }}
  </style>
</head>
<body>
  <div class="wrap">
    <section class="hero">
      <h1>AIID Executive Visual Brief</h1>
      <p class="sub">Policy-first, journalist-friendly snapshot analysis powered by AI + reproducible charts.</p>
      <p class="sub"><b>Generated:</b> {datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}</p>
    </section>

    <section class="section">
      <h2>AI Narrative Summary</h2>
      {summary_html}
    </section>

    {''.join(chart_sections)}

    <div class="foot">Responsible use note: AIID reflects documented incidents, not the full universe of AI harms.</div>
  </div>
</body>
</html>
"""

brief_path = CACHE_DIR / "AIID_Executive_Visual_Brief.html"
brief_path.write_text(report_html, encoding="utf-8")
print(f"✅ Visual brief generated: {brief_path.resolve()}")

display(HTML(f"<a href='{brief_path.name}' target='_blank' style='font-size:16px'>Open AIID_Executive_Visual_Brief.html</a>"))


## Deliverables Generated in `aiid_cache/`

- `AIID_Fact_Pack.json` — structured numeric evidence + chart insights + relationship map
- `AIID_Relationships.csv` — inferred PK/FK links across snapshot tables (if available)
- `AIID_Summary_Report.txt` / `.md` — AI narrative summary
- `AIID_Executive_Visual_Brief.html` — presentation-ready visual report
- `reports/*.html` — reusable interactive charts

### Recommended one-click flow
1. Set `SNAPSHOT_URL`, `LOCAL_ARCHIVE`, or `EXTRACTED_DIR`
2. Confirm `OPENAI_API_KEY` in environment
3. Run **Kernel → Restart & Run All**
